In [1]:
import { display } from 'tslab';

# A Crypto-Arithmetic Puzzle

In this exercise we will solve the crypto-arithmetic puzzle shown in the picture below:
<img src="send-more-money.png">

The idea is that the letters 
"$\texttt{S}$", "$\texttt{E}$", "$\texttt{N}$", "$\texttt{D}$", "$\texttt{M}$", "$\texttt{O}$", "$\texttt{R}$", "$\texttt{Y}$" occurring in this puzzle
are interpreted as variables ranging over the set of decimal digits, i.e. these variables can take values in
the set $\{0,1,2,3,4,5,6,7,8,9\}$.  Then, the string "$\texttt{SEND}$" is interpreted as a decimal number,
i.e. it is interpreted as the number
$$\texttt{S} \cdot 10^3 + \texttt{E} \cdot 10^2 + \texttt{N} \cdot 10^1 + \texttt{D} \cdot 10^0.$$
The strings "$\texttt{MORE}$ and "$\texttt{MONEY}$" are interpreted similarly. To make the problem
interesting, the assumption is that different variables have different values.  Furthermore, the
digits at the beginning of a number should be different from $0$.  Then, we have to find values for the variables
"$\texttt{S}$", "$\texttt{E}$", "$\texttt{N}$", "$\texttt{D}$", "$\texttt{M}$", "$\texttt{O}$", "$\texttt{R}$", "$\texttt{Y}$" such that the formula
$$   (\texttt{S} \cdot 10^3 + \texttt{E} \cdot 10^2 + \texttt{N} \cdot 10 + \texttt{D}) 
  + (\texttt{M} \cdot 10^3 + \texttt{O} \cdot 10^2 + \texttt{R} \cdot 10 + \texttt{E})
  = \texttt{M} \cdot 10^4 + \texttt{O} \cdot 10^3 + \texttt{N} \cdot 10^2 + \texttt{E} \cdot 10 + \texttt{Y}
$$
is true.  The problem with this constraint is that it involves far too many variables.  As this constraint can only be
checked when all the variables have values assigned to them, the backtracking search would essentially
boil down to a mere brute force search.  We would have 8 variables and hence we would have to test 
$$ 10 \cdot 9 \cdot \dots \cdot 3 = 1,814,400 $$
possible assignments. While it is certainly feasible to test this number of assignments on a modern computer, we want to do much better.  To this end we have to perform the addition in the figure shown above
column by column, just as it is taught in elementary school.  To be able to do this, we have to introduce <a href="https://en.wikipedia.org/wiki/Carry_(arithmetic)">carry digits</a> "$\texttt{C1}$", "$\texttt{C2}$", "$\texttt{C3}$" where $\texttt{C1}$ is the carry produced by adding 
$\texttt{D}$ and $\texttt{E}$, $\texttt{C2}$ is the carry produced by adding 
$\texttt{N}$, $\texttt{R}$ and $\texttt{C1}$, and $\texttt{C3}$ is the carry produced by adding 
$\texttt{E}$, $\texttt{O}$ and $\texttt{C2}$. 

In [2]:
import { CSP, Assignment, solve } from "./02-Backtracking-Constraint-Solver";

For a set $V$ of variables, the function $\texttt{allDifferent}(V)$ generates a set of formulas that express that all the variables of $V$ are different. 

In [3]:
function allDifferent(vars: string[]): string[] {
    return vars.flatMap(x => vars.filter(y => x < y).map(y => `${x} != ${y}`));
}

In [4]:
allDifferent(['a', 'b', 'c']);

[ 'a != b', 'a != c', 'b != c' ]


The function `createCSP` returns a *constraint satisfaction problem* that encodes the
cryptoarithmetic puzzle.

In [5]:
[...[1,2,3], 4, 5]

[ 1, 2, 3, 4, 5 ]


In [20]:
function createCSP(): CSP {
    const Variables   = ["D", "E", "Y", "C1", "N", "R", "C2", "O", "C3", "S", "M"];
    const Values      = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9];
    const Constraints = ["D + E == Y + 10*C1",
                        "N + R + C1 == E + 10*C2",
                        "E + O + C2 == N + 10*C3",
                        "S + M + C3 == O + 10*M",
                        "M != 0",
                        "S != 0",
                        ...allDifferent(["S", "E", "N", "D", "M", "O", "R", "Y"])];
        
    return [Variables, Values, Constraints];
}

In [21]:
const puzzle = createCSP();
puzzle;

[
  [
    'D',  'E', 'Y',  'C1',
    'N',  'R', 'C2', 'O',
    'C3', 'S', 'M'
  ],
  [
    0, 1, 2, 3, 4,
    5, 6, 7, 8, 9
  ],
  [
    'D + E == Y + 10*C1',
    'N + R + C1 == E + 10*C2',
    'E + O + C2 == N + 10*C3',
    'S + M + C3 == O + 10*M',
    'M != 0',
    'S != 0',
    'S != Y',
    'E != S',
    'E != N',
    'E != M',
    'E != O',
    'E != R',
    'E != Y',
    'N != S',
    'N != O',
    'N != R',
    'N != Y',
    'D != S',
    'D != E',
    'D != N',
    'D != M',
    'D != O',
    'D != R',
    'D != Y',
    'M != S',
    'M != N',
    'M != O',
    'M != R',
    'M != Y',
    'O != S',
    'O != R',
    'O != Y',
    'R != S',
    'R != Y'
  ]
]


In [22]:
console.time("solve");
const Solution = solve(puzzle);
console.timeEnd("solve");

solve: 165.237ms


In [23]:
Solution;

RecursiveMap(11) {
  C1 => 1,
  C2 => 1,
  C3 => 0,
  D => 7,
  E => 5,
  M => 1,
  N => 6,
  O => 0,
  R => 8,
  S => 9,
  Y => 2
}


In [24]:
function printSolution(A: Assignment | null) {
    if (A === null) {
        console.log("no solution found");
        return;
    }
    for (const v of ["S", "E", "N", "D", "M", "O", "R", "Y"]) {
        console.log(`${v} = ${A.get(v)}`);
    }
    console.log("\nThe solution of\n");
    console.log("    S E N D");
    console.log("  + M O R E");
    console.log("  ---------");
    console.log("  M O N E Y");
    console.log("\nis as follows\n");
    console.log(`    ${A.get('S')} ${A.get('E')} ${A.get('N')} ${A.get('D')}`);
    console.log(`  + ${A.get('M')} ${A.get('O')} ${A.get('R')} ${A.get('E')}`);
    console.log(`  ==========`);
    console.log(`  ${A.get('M')} ${A.get('O')} ${A.get('N')} ${A.get('E')} ${A.get('Y')}`);
}

In [25]:
printSolution(Solution);

S = 9
E = 5
N = 6
D = 7
M = 1
O = 0
R = 8
Y = 2

The solution of

    S E N D
  + M O R E
  ---------
  M O N E Y

is as follows

    9 5 6 7
  + 1 0 8 5
  1 0 6 5 2
